In [4]:
if (!require(reticulate)) {
  install.packages("reticulate")
  library(reticulate)
}
if (!require(coin)) {
  install.packages("coin")
  library(coin)
}
use_virtualenv("env/", required = TRUE)
source_python("analysis/analysis_utils.py")

set.seed(135911)

In [2]:
med_helper <- function(scores) {
  scores <- unlist(scores)
  median(scores)
}

range_helper <- function(scores) {
  scores <- unlist(scores)
  max(scores) - min(scores)
}

custom_snr <- function(median, range) {
  1 / (-(median * range))
}


preprocess_results <- function(results) {
  results$salinity <- sapply(results$salinity, factor)
  results$sorbent <- sapply(results$sorbent, factor)
  results$model <- sapply(results$model, factor)
  results$fingerprint <- sapply(results$fingerprint, factor)
  results$score.median <- sapply(results$score, med_helper)
  results$score.range <- sapply(results$score, range_helper)
  results$stat <- mapply(custom_snr, results$score.median, results$score.range)
  results$smf_inter <- with(
    results, interaction(sorbent, model, fingerprint), drop = TRUE
  )
  results$mf_inter <- with(
    results, interaction(model, fingerprint), drop = TRUE
  )
  results$stc_inter <- with(
    results, interaction(salinity, temperature, crange), drop = TRUE
  )
  results
}

all_results <- preprocess_results(load_results(
    "models",
    flatten = TRUE, r2_cutoff = 0.85, target = c("kf", "n"), extra_factor_suffices = list("emb_dim_20", "emb_dim_50"),
  )
)
all_results['filter'] <- 'normal'
print(paste("All results have dimension:", dim(all_results)[1]))
print(paste(" From which", dim(all_results[all_results$dim_red == 'normal',])[1], "are the hpo dim embeddings and",
  dim(all_results[all_results$dim_red == 'emb_dim_20',])[1], "resp",
  dim(all_results[all_results$dim_red == 'emb_dim_50',])[1],  "are from 20 resp. min(50, n_features) dim embeddings."
))

[1] "All results have dimension: 4160"
[1] " From which 1600 are the hpo dim embeddings and 1280 resp 1280 are from 20 resp. min(50, n_features) dim embeddings."


In [5]:
oom_results <- preprocess_results(
  load_results(
    "models/has_enough_ooms", flatten = TRUE, r2_cutoff = 0.85, target = c('kf', 'n'),
  )
)
oom_results['filter'] <- 'ooms'
oom_results['dim_red'] <- 'normal'
print(paste("Results for enough ooms have dimension:", dim(oom_results)[1]))
print(paste(" With", paste(c(unique(oom_results$salinity)), collapse = ", "), "as water selection."))

lin_results <- preprocess_results(
  load_results(
    "models/is_linear_fit", flatten = TRUE, r2_cutoff = 0.85, target = c('kf', 'n'),
  )
)
lin_results['filter'] <- 'linear fit'
lin_results['dim_red'] <- 'normal'
print(paste("Results for linear fit have dimension:", dim(lin_results)[1]))
print(paste(" With", paste(c(unique(lin_results$salinity)), collapse = ", "), "as water selection."))

nlin_results <- preprocess_results(
  load_results(
    "models/is_non_linear_fit", flatten = TRUE, r2_cutoff = 0.85, target = c('kf', 'n'),
  )
)
nlin_results['filter'] <- 'non-linear fit'
nlin_results['dim_red'] <- 'normal'
print(paste("Results for non-linear fit have dimension:", dim(nlin_results)[1]))
print(paste(" With", paste(c(unique(nlin_results$salinity)), collapse = ", "), "as water selection."))

isofit_results <- preprocess_results(
  load_results(
    "models/iso_fit_as_bool", flatten = TRUE, r2_cutoff = 0.85, target = c('kf', 'n'),
  )
)
isofit_results['filter'] <- 'iso_fit_as_bool'
isofit_results['dim_red'] <- 'normal'
print(paste("Results for isotherm fit as boolean have dimension:", dim(isofit_results)[1]))
print(paste(" With", paste(c(unique(isofit_results$salinity)), collapse = ", "), "as water selection."))

oom_isofit_results <- preprocess_results(
  load_results(
    "models/ooms+fit_as_bool", flatten = TRUE, r2_cutoff = 0.85, target = c('kf', 'n'),
  )
)
oom_isofit_results['filter'] <- 'ooms+fit_as_bool'
oom_isofit_results['dim_red'] <- 'normal'
print(paste("Results for isotherm fit as boolean have dimension:", dim(oom_isofit_results)[1]))
print(paste(" With", paste(c(unique(oom_isofit_results$salinity)), collapse = ", "), "as water selection."))

[1] "Results for enough ooms have dimension: 1600"
[1] " With freshwater, seawater, all_water, all_vals as water selection."
[1] "Results for linear fit have dimension: 1200"
[1] " With seawater, all_water, all_vals as water selection."
[1] "Results for non-linear fit have dimension: 400"
[1] " With freshwater as water selection."
[1] "Results for isotherm fit as boolean have dimension: 1600"
[1] " With freshwater, seawater, all_water, all_vals as water selection."
[1] "Results for isotherm fit as boolean have dimension: 1600"
[1] " With freshwater, seawater, all_water, all_vals as water selection."


In [6]:
all_results <- rbind(all_results, oom_results, lin_results, nlin_results, isofit_results, oom_isofit_results)
all_results$filter <-sapply(all_results$filter, factor)
all_results$dim_red <-sapply(all_results$dim_red, factor)
print((paste("Having a total of", dim(all_results)[1], "models.")))

[1] "Having a total of 10560 models."


# Permutation Tests

## Normal settings
compare all models, fingerprints and sorbent representation as well as inclusion/exclusion of salinity, concentration range and temperature for kf and n

In [7]:
normal_results <- all_results[
  (all_results$dim_red == 'normal') & (all_results$filter == 'normal'), 
]
print(dim(normal_results))

print("Results for stat ~ model + sorbent + fingerprint | stc_inter")
it <- independence_test(
  stat ~ model + sorbent + fingerprint | stc_inter,
  data=normal_results[normal_results$target == 'kf', ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))

it <- independence_test(
  stat ~ model + sorbent + fingerprint | stc_inter,
  data=normal_results[normal_results$target == 'kf', ],
  distribution = approximate(nresample = 1000000),
  alternative = "less"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))


it <- independence_test(
  stat ~ model + sorbent + fingerprint | stc_inter,
  data=normal_results[normal_results$target == 'n', ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))
it <- independence_test(
  stat ~ model + sorbent + fingerprint | stc_inter,
  data=normal_results[normal_results$target == 'n', ],
  distribution = approximate(nresample = 1000000),
  alternative = "less"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))

[1] 1600   16
[1] "Results for stat ~ model + sorbent + fingerprint | stc_inter"


[1] "Results for kf"

	Approximative General Independence Test

data:  stat by
	 model, sorbent, fingerprint 
	 stratified by stc_inter
maxT = 11.205, p-value < 1e-06
alternative hypothesis: two.sided

                              
model.rf              0.049962
model.gp              0.031529
model.pls            <0.000001
model.krr            <0.000001
model.mlp             0.049962
sorbent.name          0.961611
fingerprint.mol2vec   0.692401
fingerprint.maccs    <0.000001
fingerprint.pubchem   0.961611
fingerprint.rdkit    <0.000001
fingerprint.abrahams  0.961611
[1] "Results for kf"

	Approximative General Independence Test

data:  stat by
	 model, sorbent, fingerprint 
	 stratified by stc_inter
maxT = -11.205, p-value < 1e-06
alternative hypothesis: less

                              
model.rf              1.000000
model.gp              1.000000
model.pls            <0.000001
model.krr             1.000000
model.mlp             0.032622
sorbent.name          0.966898
fingerprint

PLS and MLP models, as well as RDKit Fingerprint produce significantly worse results. From visual inspection of the rmse error distributions we see, that pls consistently produces worse models and the other factors are neither so consistent nor so huge in difference -> drop only pls from further analysis, with the reasoning, that this is from a different distribution, hence skews the empirical distribution whereas MLP and RDKit are only bad results but from similar distribution.

In [8]:
all_results_filt <- all_results[all_results$model != 'pls',]

In [9]:
normal_results <- all_results_filt[
  (all_results_filt$dim_red == 'normal') & (all_results_filt$filter == 'normal'), 
]
print(dim(normal_results))

print("Results for stat ~ model + sorbent + fingerprint | stc_inter")
it <- independence_test(
  stat ~ model + sorbent + fingerprint | stc_inter,
  data=normal_results[normal_results$target == 'kf', ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))

it <- independence_test(
  stat ~ model + sorbent + fingerprint | stc_inter,
  data=normal_results[normal_results$target == 'kf', ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))


it <- independence_test(
  stat ~ model + sorbent + fingerprint | stc_inter,
  data=normal_results[normal_results$target == 'n', ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))
it <- independence_test(
  stat ~ model + sorbent + fingerprint | stc_inter,
  data=normal_results[normal_results$target == 'n', ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))

[1] 1280   16
[1] "Results for stat ~ model + sorbent + fingerprint | stc_inter"


[1] "Results for kf"

	Approximative General Independence Test

data:  stat by
	 model, sorbent, fingerprint 
	 stratified by stc_inter
maxT = 6.2323, p-value < 1e-06
alternative hypothesis: two.sided

                              
model.rf              0.955574
model.gp              0.979966
model.krr            <0.000001
model.mlp            <0.000001
sorbent.name          0.940338
fingerprint.mol2vec   0.801760
fingerprint.maccs     0.000004
fingerprint.pubchem   0.940338
fingerprint.rdkit    <0.000001
fingerprint.abrahams  0.375185
[1] "Results for kf"

	Approximative General Independence Test

data:  stat by
	 model, sorbent, fingerprint 
	 stratified by stc_inter
maxT = 6.2003, p-value < 1e-06
alternative hypothesis: greater

                              
model.rf              0.994478
model.gp              0.994123
model.krr            <0.000001
model.mlp             1.000000
sorbent.name          0.995924
fingerprint.mol2vec   0.998737
fingerprint.maccs     0.000008
fingerpri

Restricting to KRR and RF models

In [10]:
normal_results <- all_results_filt[
  (all_results_filt$dim_red == 'normal') & (all_results_filt$filter == 'normal'), 
]
print(dim(normal_results))

print("Results for stat ~ sorbent + fingerprint | stc_inter")
it <- independence_test(
  stat ~ sorbent + fingerprint | stc_inter,
  data=normal_results[(normal_results$target == 'kf') & (normal_results$model == 'krr'), ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for kf and KRR only")
print(it)
print(pvalue(it, method = "step-down"))

it <- independence_test(
  stat ~ sorbent + fingerprint | stc_inter,
  data=normal_results[(normal_results$target == 'kf') & (normal_results$model == 'krr'), ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))


it <- independence_test(
  stat ~ sorbent + fingerprint | stc_inter,
  data=normal_results[(normal_results$target == 'n') & (normal_results$model == 'rf'), ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for n and RF only")
print(it)
print(pvalue(it, method = "step-down"))
it <- independence_test(
  stat ~ sorbent + fingerprint | stc_inter,
  data=normal_results[(normal_results$target == 'n') & (normal_results$model == 'rf'), ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))

[1] 1280   16
[1] "Results for stat ~ sorbent + fingerprint | stc_inter"


[1] "Results for kf and KRR only"

	Approximative General Independence Test

data:  stat by
	 sorbent, fingerprint 
	 stratified by stc_inter
maxT = 3.3743, p-value = 0.004148
alternative hypothesis: two.sided

                             
sorbent.name         0.901483
fingerprint.mol2vec  0.903453
fingerprint.maccs    0.004148
fingerprint.pubchem  0.034676
fingerprint.rdkit    0.138423
fingerprint.abrahams 0.362001
[1] "Results for kf"

	Approximative General Independence Test

data:  stat by
	 sorbent, fingerprint 
	 stratified by stc_inter
maxT = 3.3743, p-value = 0.003955
alternative hypothesis: greater

                             
sorbent.name         0.848958
fingerprint.mol2vec  0.948834
fingerprint.maccs    0.003955
fingerprint.pubchem  0.999990
fingerprint.rdkit    0.999990
fingerprint.abrahams 0.358361
[1] "Results for n and RF only"

	Approximative General Independence Test

data:  stat by
	 sorbent, fingerprint 
	 stratified by stc_inter
maxT = 2.1385, p-value = 0.1635
a

## Results regarding salinity and inclusion/exclusion of concentration range and temperature

In [11]:
normal_results <- all_results_filt[
  (all_results_filt$dim_red == 'normal') & (all_results_filt$filter == 'normal'), 
]
print(dim(normal_results))

print("Results for stat ~ salinity + temperature + crange | mf_inter")
it <- independence_test(
  stat ~ salinity + temperature + crange | mf_inter,
  data=normal_results[normal_results$target == 'kf', ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))

it <- independence_test(
  stat ~ salinity + temperature + crange | mf_inter,
  data=normal_results[normal_results$target == 'kf', ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))


it <- independence_test(
  stat ~ salinity + temperature + crange | mf_inter,
  data=normal_results[normal_results$target == 'n', ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))
it <- independence_test(
  stat ~ salinity + temperature + crange | mf_inter,
  data=normal_results[normal_results$target == 'n', ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))

[1] 1280   16
[1] "Results for stat ~ salinity + temperature + crange | mf_inter"


[1] "Results for kf"

	Approximative General Independence Test

data:  stat by
	 salinity, temperature, crange 
	 stratified by mf_inter
maxT = 14.696, p-value < 1e-06
alternative hypothesis: two.sided

                             
salinity.freshwater  0.016762
salinity.seawater   <0.000001
salinity.all_water  <0.000001
salinity.all_vals   <0.000001
temperature         <0.000001
crange               0.046022
[1] "Results for kf"

	Approximative General Independence Test

data:  stat by
	 salinity, temperature, crange 
	 stratified by mf_inter
maxT = 7.3301, p-value < 1e-06
alternative hypothesis: greater

                             
salinity.freshwater  0.013526
salinity.seawater    1.000000
salinity.all_water  <0.000001
salinity.all_vals   <0.000001
temperature         <0.000001
crange               0.999503
[1] "Results for n"

	Approximative General Independence Test

data:  stat by
	 salinity, temperature, crange 
	 stratified by mf_inter
maxT = 14.31, p-value < 1e-06
alternativ

Restricting to KRR and RF

In [12]:
normal_results <- all_results_filt[
  (all_results_filt$dim_red == 'normal') & (all_results_filt$filter == 'normal'), 
]
print(dim(normal_results))

print("Results for stat ~ salinity + temperature + crange | mf_inter")
it <- independence_test(
  stat ~ salinity + temperature + crange | fingerprint,
  data=normal_results[(normal_results$target == 'kf') & (normal_results$model == 'krr'), ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for kf and KRR only")
print(it)
print(pvalue(it, method = "step-down"))

it <- independence_test(
  stat ~ salinity + temperature + crange | fingerprint,
  data=normal_results[(normal_results$target == 'kf') & (normal_results$model == 'krr'), ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))


it <- independence_test(
  stat ~ salinity + temperature + crange | fingerprint,
  data=normal_results[(normal_results$target == 'n') & (normal_results$model == 'rf'), ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for n and RF only")
print(it)
print(pvalue(it, method = "step-down"))
it <- independence_test(
  stat ~ salinity + temperature + crange | fingerprint,
  data=normal_results[(normal_results$target == 'n') & (normal_results$model == 'rf'), ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))

[1] 1280   16
[1] "Results for stat ~ salinity + temperature + crange | mf_inter"


[1] "Results for kf and KRR only"

	Approximative General Independence Test

data:  stat by
	 salinity, temperature, crange 
	 stratified by fingerprint
maxT = 7.6086, p-value < 1e-06
alternative hypothesis: two.sided

                             
salinity.freshwater  0.038941
salinity.seawater   <0.000001
salinity.all_water   0.077156
salinity.all_vals    0.006452
temperature         <0.000001
crange               0.446354
[1] "Results for kf"

	Approximative General Independence Test

data:  stat by
	 salinity, temperature, crange 
	 stratified by fingerprint
maxT = 5.6154, p-value < 1e-06
alternative hypothesis: greater

                             
salinity.freshwater  0.031830
salinity.seawater    1.000000
salinity.all_water   0.064057
salinity.all_vals    0.005939
temperature         <0.000001
crange               0.950081
[1] "Results for n and RF only"

	Approximative General Independence Test

data:  stat by
	 salinity, temperature, crange 
	 stratified by fingerprint
maxT =

## Comparing dimension reduction strategies

In [13]:
dim_red_results <- all_results_filt[all_results_filt$filter == 'normal',]
print(dim(dim_red_results))

print("Results for stat ~ model + sorbent + fingerprint + dim_red | stc_inter")
it <- independence_test(
  stat ~  model + sorbent + fingerprint + dim_red | stc_inter,
  data=dim_red_results[dim_red_results$target == 'kf', ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))

it <- independence_test(
  stat ~ model + sorbent + fingerprint + dim_red | stc_inter,
  data=dim_red_results[dim_red_results$target == 'kf', ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))

it <- independence_test(
  stat ~ model + sorbent + fingerprint + dim_red | stc_inter,
  data=dim_red_results[dim_red_results$target == 'n', ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))
it <- independence_test(
  stat ~ model + sorbent + fingerprint + dim_red | stc_inter,
  data=dim_red_results[dim_red_results$target == 'n', ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))

print("The same with the other block structure:")
print("Results for stat ~ salinity + temperature + crange + dim_red | mf_inter")
it <- independence_test(
  stat ~ salinity + temperature + crange + dim_red | mf_inter,
  data=dim_red_results[dim_red_results$target == 'kf', ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))

it <- independence_test(
  stat ~ salinity + temperature + crange + dim_red | mf_inter,
  data=dim_red_results[dim_red_results$target == 'kf', ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))


it <- independence_test(
  stat ~ salinity + temperature + crange + dim_red | mf_inter,
  data=dim_red_results[dim_red_results$target == 'n', ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))
it <- independence_test(
  stat ~ salinity + temperature + crange + dim_red | mf_inter,
  data=dim_red_results[dim_red_results$target == 'n', ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))

[1] 3328   16
[1] "Results for stat ~ model + sorbent + fingerprint + dim_red | stc_inter"


[1] "Results for kf"

	Approximative General Independence Test

data:  stat by
	 model, sorbent, fingerprint, dim_red 
	 stratified by stc_inter
maxT = 9.6238, p-value < 1e-06
alternative hypothesis: two.sided

                              
model.rf              0.718444
model.gp              0.389336
model.krr            <0.000001
model.mlp            <0.000001
sorbent.name          0.930461
fingerprint.mol2vec   0.026028
fingerprint.maccs    <0.000001
fingerprint.pubchem  <0.000001
fingerprint.rdkit    <0.000001
fingerprint.abrahams  0.930461
dim_red.normal        0.001153
dim_red.emb_dim_20    0.930461
dim_red.emb_dim_50    0.000080
[1] "Results for kf"

	Approximative General Independence Test

data:  stat by
	 model, sorbent, fingerprint, dim_red 
	 stratified by stc_inter
maxT = 6.965, p-value < 1e-06
alternative hypothesis: greater

                              
model.rf              0.999994
model.gp              0.369216
model.krr            <0.000001
model.mlp             1

Restricting to KRR and RF

In [14]:
dim_red_results <- all_results_filt[all_results_filt$filter == 'normal',]
print(dim(dim_red_results))

print("Results for stat ~ model + sorbent + fingerprint + dim_red | stc_inter")
it <- independence_test(
  stat ~  sorbent + fingerprint + dim_red | stc_inter,
  data=dim_red_results[(dim_red_results$target == 'kf') & (dim_red_results$model == 'krr'), ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for kf and KRR only")
print(it)
print(pvalue(it, method = "step-down"))

it <- independence_test(
  stat ~ sorbent + fingerprint + dim_red | stc_inter,
  data=dim_red_results[(dim_red_results$target == 'kf') & (dim_red_results$model == 'krr'), ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))

it <- independence_test(
  stat ~ sorbent + fingerprint + dim_red | stc_inter,
  data=dim_red_results[(dim_red_results$target == 'n') & (dim_red_results$model == 'rf'), ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for n and RF only")
print(it)
print(pvalue(it, method = "step-down"))
it <- independence_test(
  stat ~ sorbent + fingerprint + dim_red | stc_inter,
  data=dim_red_results[(dim_red_results$target == 'n') & (dim_red_results$model == 'rf'), ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))


print("The same with the other block structure:")
print("Results for stat ~ salinity + temperature + crange + dim_red | mf_inter")
it <- independence_test(
  stat ~ salinity + temperature + crange + dim_red | mf_inter,
  data=dim_red_results[(dim_red_results$target == 'kf') & (dim_red_results$model == 'krr'), ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))

it <- independence_test(
  stat ~ salinity + temperature + crange + dim_red | mf_inter,
  data=dim_red_results[(dim_red_results$target == 'kf') & (dim_red_results$model == 'krr'), ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))


it <- independence_test(
  stat ~ salinity + temperature + crange + dim_red | mf_inter,
  data=dim_red_results[(dim_red_results$target == 'n') & (dim_red_results$model == 'rf'), ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))
it <- independence_test(
  stat ~ salinity + temperature + crange + dim_red | mf_inter,
  data=dim_red_results[(dim_red_results$target == 'n') & (dim_red_results$model == 'rf'), ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))

[1] 3328   16
[1] "Results for stat ~ model + sorbent + fingerprint + dim_red | stc_inter"


[1] "Results for kf and KRR only"

	Approximative General Independence Test

data:  stat by
	 sorbent, fingerprint, dim_red 
	 stratified by stc_inter
maxT = 4.8095, p-value = 1.5e-05
alternative hypothesis: two.sided

                             
sorbent.name         0.634538
fingerprint.mol2vec  0.939693
fingerprint.maccs    0.000151
fingerprint.pubchem  0.865837
fingerprint.rdkit    0.000015
fingerprint.abrahams 0.634538
dim_red.normal       0.939693
dim_red.emb_dim_20   0.816360
dim_red.emb_dim_50   0.738821
[1] "Results for kf"

	Approximative General Independence Test

data:  stat by
	 sorbent, fingerprint, dim_red 
	 stratified by stc_inter
maxT = 4.2772, p-value = 0.000153
alternative hypothesis: greater

                             
sorbent.name         0.498741
fingerprint.mol2vec  0.954034
fingerprint.maccs    0.000153
fingerprint.pubchem  0.990153
fingerprint.rdkit    1.000000
fingerprint.abrahams 0.498741
dim_red.normal       0.986432
dim_red.emb_dim_20   0.990153
dim_re

## Isotherm filtering strategies

In [15]:
unique(all_results$filter)

[1] normal           ooms             linear fit       non-linear fit  
[5] iso_fit_as_bool  ooms+fit_as_bool
6 Levels: normal ooms linear fit non-linear fit ... ooms+fit_as_bool

Seawater (normal vs linear fit vs ooms)

In [16]:
seawater_results <- all_results_filt[
  (all_results_filt$dim_red == 'normal') &
  (all_results_filt$filter %in% c('normal', 'linear fit', 'ooms')) &
  (all_results_filt$salinity == 'seawater'),]
print(dim(seawater_results))
seawater_results$tcmf_inter <- with(
    seawater_results, interaction(temperature, crange, model, fingerprint), drop = TRUE
  )
seawater_results$tc_inter <- with(
    seawater_results, interaction(temperature, crange), drop = TRUE
  )

############### Kf
print("temperature + crange + filter | mf_inter")
it <- independence_test(
  stat ~ temperature + crange + filter | mf_inter,
  data=seawater_results[seawater_results$target == 'kf', ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))

it <- independence_test(
  stat ~ temperature + crange + filter | mf_inter,
  data=seawater_results[seawater_results$target == 'kf', ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))

############### n

it <- independence_test(
  stat ~ temperature + crange + filter | mf_inter,
  data=seawater_results[seawater_results$target == 'n', ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))
it <- independence_test(
  stat ~ temperature + crange + filter | mf_inter,
  data=seawater_results[seawater_results$target == 'n', ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))

#########################
print("With the other block structure:")
############### Kf
print("model + fingerprint + filter | tc_inter")
it <- independence_test(
  stat ~ model + fingerprint + filter | tc_inter,
  data=seawater_results[seawater_results$target == 'kf', ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))

it <- independence_test(
  stat ~ model + fingerprint + filter | tc_inter,
  data=seawater_results[seawater_results$target == 'kf', ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))

############### n

it <- independence_test(
  stat ~ model + fingerprint + filter | tc_inter,
  data=seawater_results[seawater_results$target == 'n', ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))
it <- independence_test(
  stat ~ model + fingerprint + filter | tc_inter,
  data=seawater_results[seawater_results$target == 'n', ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))

[1] 960  16
[1] "temperature + crange + filter | mf_inter"


[1] "Results for kf"

	Approximative General Independence Test

data:  stat by
	 temperature, crange, filter 
	 stratified by mf_inter
maxT = 9.3053, p-value < 1e-06
alternative hypothesis: two.sided

                           
temperature        0.000019
crange             0.075932
filter.normal      0.000012
filter.ooms        0.000004
filter.linear fit <0.000001
[1] "Results for kf"

	Approximative General Independence Test

data:  stat by
	 temperature, crange, filter 
	 stratified by mf_inter
maxT = 9.3053, p-value < 1e-06
alternative hypothesis: greater

                        
temperature        1e+00
crange             1e+00
filter.normal      1e+00
filter.ooms        1e+00
filter.linear fit <1e-06
[1] "Results for n"

	Approximative General Independence Test

data:  stat by
	 temperature, crange, filter 
	 stratified by mf_inter
maxT = 4.1931, p-value = 0.000106
alternative hypothesis: two.sided

                          
temperature       0.661796
crange            0.66179

freshwater (non-linear fit vs ooms vs normal)

In [21]:
freshwater_results <- all_results_filt[
  (all_results_filt$dim_red == 'normal') &
  !(all_results_filt$filter %in% c('linear fit', 'iso_fit_as_bool', 'ooms+fit_as_bool')) &
  (all_results_filt$salinity == 'freshwater'),]
print(dim(freshwater_results))
freshwater_results$tc_inter <- with(
    freshwater_results, interaction(temperature, crange), drop = TRUE
  )

############### Kf
print("Results for stat ~ temperature + crange + filter | mf_inter")
it <- independence_test(
  stat ~ temperature + crange + filter | mf_inter,
  data=freshwater_results[freshwater_results$target == 'kf', ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))

it <- independence_test(
  stat ~ temperature + crange + filter | mf_inter,
  data=freshwater_results[freshwater_results$target == 'kf', ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))

############### n
it <- independence_test(
  stat ~ temperature + crange + filter | mf_inter,
  data=freshwater_results[freshwater_results$target == 'n', ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))
it <- independence_test(
  stat ~ temperature + crange + filter | mf_inter,
  data=freshwater_results[freshwater_results$target == 'n', ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))

####################
print("With other block structure:")
############### Kf
print("Results for stat ~ model + fingerprint + filter | tc_inter")
it <- independence_test(
  stat ~ model + fingerprint + filter | tc_inter,
  data=freshwater_results[freshwater_results$target == 'kf', ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))

it <- independence_test(
  stat ~ model + fingerprint + filter | tc_inter,
  data=freshwater_results[freshwater_results$target == 'kf', ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))

############### n
it <- independence_test(
  stat ~ model + fingerprint + filter | tc_inter,
  data=freshwater_results[freshwater_results$target == 'n', ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))
it <- independence_test(
  stat ~ model + fingerprint + filter | tc_inter,
  data=freshwater_results[freshwater_results$target == 'n', ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))

[1] 960  16
[1] "Results for stat ~ temperature + crange + filter | mf_inter"


[1] "Results for kf"

	Approximative General Independence Test

data:  stat by
	 temperature, crange, filter 
	 stratified by mf_inter
maxT = 3.4302, p-value = 0.0027
alternative hypothesis: two.sided

                              
temperature           0.002700
crange                0.123408
filter.normal         0.002742
filter.ooms           0.017028
filter.non-linear fit 0.537907
[1] "Results for kf"

	Approximative General Independence Test

data:  stat by
	 temperature, crange, filter 
	 stratified by mf_inter
maxT = 3.4302, p-value = 0.001638
alternative hypothesis: greater

                              
temperature           0.001638
crange                0.999042
filter.normal         0.999724
filter.ooms           0.012374
filter.non-linear fit 0.643818
[1] "Results for n"

	Approximative General Independence Test

data:  stat by
	 temperature, crange, filter 
	 stratified by mf_inter
maxT = 11.517, p-value < 1e-06
alternative hypothesis: two.sided

                        

All water (normal vs ooms vs isotherm_fit_as_bool)

In [18]:
all_water_results <- all_results_filt[
  (all_results_filt$dim_red == 'normal') &
  (all_results_filt$filter %in% c('normal', 'ooms', 'iso_fit_as_bool', 'ooms+fit_as_bool')) &
  (all_results_filt$salinity %in% c('all_water', 'all_vals')),]
print(dim(all_water_results))
all_water_results$tc_inter <- with(
    all_water_results, interaction(temperature, crange), drop = TRUE
  )

############### Kf
print("Results for stat ~ temperature + salinity + crange + filter | mf_inter")
it <- independence_test(
  stat ~ temperature + salinity + crange + filter | mf_inter,
  data=all_water_results[all_water_results$target == 'kf', ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))

it <- independence_test(
  stat ~ temperature + salinity + crange + filter | mf_inter,
  data=all_water_results[all_water_results$target == 'kf', ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))

############### n
it <- independence_test(
  stat ~ temperature + salinity + crange + filter | mf_inter,
  data=all_water_results[all_water_results$target == 'n', ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))
it <- independence_test(
  stat ~ temperature + salinity + crange + filter | mf_inter,
  data=all_water_results[all_water_results$target == 'n', ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))

###################
print("With other block structure:")
############### Kf
print("Results for stat ~ model + fingerprint + filter | tc_inter")
it <- independence_test(
  stat ~ model + fingerprint + filter | tc_inter,
  data=all_water_results[all_water_results$target == 'kf', ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))

it <- independence_test(
  stat ~ model+ fingerprint + filter | tc_inter,
  data=all_water_results[all_water_results$target == 'kf', ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))

############### n
it <- independence_test(
  stat ~ model+ fingerprint + filter | tc_inter,
  data=all_water_results[all_water_results$target == 'n', ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))
it <- independence_test(
  stat ~ model+ fingerprint + filter | tc_inter,
  data=all_water_results[all_water_results$target == 'n', ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))

[1] 2560   16
[1] "Results for stat ~ temperature + salinity + crange + filter | mf_inter"


[1] "Results for kf"

	Approximative General Independence Test

data:  stat by
	 temperature, salinity, crange, filter 
	 stratified by mf_inter
maxT = 4.7472, p-value = 1.1e-05
alternative hypothesis: two.sided

                                
temperature             0.000011
salinity.all_water      0.833126
crange                  0.022191
filter.normal           0.000109
filter.ooms             0.000050
filter.iso_fit_as_bool  0.000237
filter.ooms+fit_as_bool 0.000478
[1] "Results for kf"

	Approximative General Independence Test

data:  stat by
	 temperature, salinity, crange, filter 
	 stratified by mf_inter
maxT = 4.7472, p-value = 1e-05
alternative hypothesis: greater

                                
temperature             0.000010
salinity.all_water      0.900311
crange                  1.000000
filter.normal           1.000000
filter.ooms             0.000049
filter.iso_fit_as_bool  1.000000
filter.ooms+fit_as_bool 0.000520
[1] "Results for n"

	Approximative General Indepe

Post-Hoc comparison of normal and isotherm_fit_as_bool, as both perform significantly worse than models on ooms filtered data.

In [19]:
all_water_results <- all_results_filt[
  (all_results_filt$dim_red == 'normal') &
  (all_results_filt$filter %in% c('normal', 'iso_fit_as_bool')) &
  (all_results_filt$salinity %in% c('all_water', 'all_vals')),]
all_water_results$tc_inter <- with(
    all_water_results, interaction(temperature, crange), drop = TRUE
  )
print(dim(all_water_results))

############### Kf
print("Results for stat ~ temperature+ crange + filter | mf_inter")
it <- independence_test(
  stat ~ temperature + salinity + crange + filter | mf_inter,
  data=all_water_results[all_water_results$target == 'kf', ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))

it <- independence_test(
  stat ~ temperature + salinity + crange + filter | mf_inter,
  data=all_water_results[all_water_results$target == 'kf', ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))

############### n
it <- independence_test(
  stat ~ temperature + salinity + crange + filter | mf_inter,
  data=all_water_results[all_water_results$target == 'n', ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))
it <- independence_test(
  stat ~ temperature + salinity + crange + filter | mf_inter,
  data=all_water_results[all_water_results$target == 'n', ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))

###################
print("With other block structure:")
############### Kf
print("Results for stat ~ model + fingerprint + filter | tc_inter")
it <- independence_test(
  stat ~ model+ fingerprint + filter | tc_inter,
  data=all_water_results[all_water_results$target == 'kf', ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))

it <- independence_test(
  stat ~ model+ fingerprint + filter | tc_inter,
  data=all_water_results[all_water_results$target == 'kf', ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))

############### n
it <- independence_test(
  stat ~ model+ fingerprint + filter | tc_inter,
  data=all_water_results[all_water_results$target == 'n', ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))
it <- independence_test(
  stat ~ model+ fingerprint + filter | tc_inter,
  data=all_water_results[all_water_results$target == 'n', ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))

[1] 1280   17
[1] "Results for stat ~ temperature+ crange + filter | mf_inter"


[1] "Results for kf"

	Approximative General Independence Test

data:  stat by
	 temperature, salinity, crange, filter 
	 stratified by mf_inter
maxT = 12.45, p-value < 1e-06
alternative hypothesis: two.sided

                            
temperature        <0.000001
salinity.all_water  0.960050
crange              0.008986
filter.normal       0.960050
[1] "Results for kf"

	Approximative General Independence Test

data:  stat by
	 temperature, salinity, crange, filter 
	 stratified by mf_inter
maxT = 12.45, p-value < 1e-06
alternative hypothesis: greater

                            
temperature        <0.000001
salinity.all_water  0.784493
crange              0.998442
filter.normal       0.807497
[1] "Results for n"

	Approximative General Independence Test

data:  stat by
	 temperature, salinity, crange, filter 
	 stratified by mf_inter
maxT = 14.77, p-value < 1e-06
alternative hypothesis: two.sided

                            
temperature        <0.000001
salinity.all_water  0.835

Post-Hoc comparison of ooms and ooms with isotherm_fit_as_bool, as both perform significantly better than the other models data.

In [20]:
all_water_results <- all_results_filt[
  (all_results_filt$dim_red == 'normal') &
  (all_results_filt$filter %in% c('normal', 'iso_fit_as_bool')) &
  (all_results_filt$salinity %in% c('all_water', 'all_vals')),]
all_water_results$tc_inter <- with(
    all_water_results, interaction(temperature, crange), drop = TRUE
  )
print(dim(all_water_results))

############### Kf
print("Results for stat ~ temperature+ crange + filter | mf_inter")
it <- independence_test(
  stat ~ temperature + salinity + crange + filter | mf_inter,
  data=all_water_results[all_water_results$target == 'kf', ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))

it <- independence_test(
  stat ~ temperature + salinity + crange + filter | mf_inter,
  data=all_water_results[all_water_results$target == 'kf', ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))

############### n
it <- independence_test(
  stat ~ temperature + salinity + crange + filter | mf_inter,
  data=all_water_results[all_water_results$target == 'n', ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))
it <- independence_test(
  stat ~ temperature + salinity + crange + filter | mf_inter,
  data=all_water_results[all_water_results$target == 'n', ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))

###################
print("With other block structure:")
############### Kf
print("Results for stat ~ model+ fingerprint + filter | tc_inter")
it <- independence_test(
  stat ~ model+ fingerprint + filter | tc_inter,
  data=all_water_results[all_water_results$target == 'kf', ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))

it <- independence_test(
  stat ~ model+ fingerprint + filter | tc_inter,
  data=all_water_results[all_water_results$target == 'kf', ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for kf")
print(it)
print(pvalue(it, method = "step-down"))

############### n
it <- independence_test(
  stat ~ model+ fingerprint + filter | tc_inter,
  data=all_water_results[all_water_results$target == 'n', ],
  distribution = approximate(nresample = 1000000),
  alternative = "two.sided"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))
it <- independence_test(
  stat ~ model+ fingerprint + filter | tc_inter,
  data=all_water_results[all_water_results$target == 'n', ],
  distribution = approximate(nresample = 1000000),
  alternative = "greater"
)
print("Results for n")
print(it)
print(pvalue(it, method = "step-down"))

[1] 1280   17
[1] "Results for stat ~ temperature+ crange + filter | mf_inter"


[1] "Results for kf"

	Approximative General Independence Test

data:  stat by
	 temperature, salinity, crange, filter 
	 stratified by mf_inter
maxT = 12.45, p-value < 1e-06
alternative hypothesis: two.sided

                            
temperature        <0.000001
salinity.all_water  0.960300
crange              0.009124
filter.normal       0.960300
[1] "Results for kf"

	Approximative General Independence Test

data:  stat by
	 temperature, salinity, crange, filter 
	 stratified by mf_inter
maxT = 12.45, p-value < 1e-06
alternative hypothesis: greater

                            
temperature        <0.000001
salinity.all_water  0.784794
crange              0.998438
filter.normal       0.807540
[1] "Results for n"

	Approximative General Independence Test

data:  stat by
	 temperature, salinity, crange, filter 
	 stratified by mf_inter
maxT = 14.77, p-value < 1e-06
alternative hypothesis: two.sided

                            
temperature        <0.000001
salinity.all_water  0.836